In [1]:
import glob
import yaml
import ml_collections
from tqdm import tqdm
from utils.utils import *
from utils.metric import *
from utils.eval_portfolio.summary import *
from utils.eval_portfolio.strategies import log_return_to_price
from utils.file_manager import LOCAL_BASE_DIRECTORY, quantDiffusionPath
from models.modernTCN_cross_attention import ModernTCN
from torch.utils.data import DataLoader, TensorDataset
from modules.diffusion import TimeSeriesGaussianDiffusion, generate_cosine_schedule


file_manager = quantDiffusionPath()
gc.collect()

0

# Preprocessing

In [2]:
def fit_transform_garch_12d(df_log_returns, df_rv, rv_window=20, min_periods=5):
    eps = 1e-8
    
    # --- 핵심 수정: 인덱스 동기화 ---
    # 두 데이터프레임의 공통 인덱스만 추출
    common_idx = df_log_returns.index.intersection(df_rv.index)
    df_log_returns = df_log_returns.loc[common_idx]
    df_rv = df_rv.loc[common_idx]

    T, C = df_log_returns.shape
    print(f"Synced Shape - Returns: {df_log_returns.shape}, RV: {df_rv.shape}")

    # 1. 미래 윈도우 기반 RV 산출
    future_rv = np.sqrt(
        df_rv.rolling(rv_window, min_periods=min_periods)
        .mean()
        .shift(-(rv_window - 1))
    )

    # 2. Bulk(Z) 및 Tail(V) 분해
    # 인덱스가 맞춰졌으므로 이제 안전하게 나눗셈 가능
    Z_t = (df_log_returns / (future_rv + eps))
    V_t = np.log(future_rv + eps)
    
    # 3. 결측치 제거 
    # (수익률 계산 시 발생한 NaN + Future window로 인한 끝부분 NaN 통합 제거)
    combined = pd.concat([Z_t, V_t], axis=1).dropna()
    
    # 차원 재확인
    Z_t_clean = combined.iloc[:, :C]
    V_t_clean = combined.iloc[:, C:]
    print(f"Final Cleaned Shape: {combined.shape}")

    # 4. 12차원 결합 및 스케일링
    scaled_12d_raw = combined.values
    scaled_12d, scalers = scaling(scaled_12d_raw)
    
    # 5. 직교성 점수 확인
    corrs = []
    for i in range(C):
        c = np.corrcoef(Z_t_clean.iloc[:, i], V_t_clean.iloc[:, i])[0, 1]
        corrs.append(c)
    
    avg_orthogonality = np.mean(np.abs(corrs))
    print(f"Average Orthogonality Score (Corr): {avg_orthogonality:.4f}")
    scaled_12d_raw = pd.concat([Z_t, np.log(future_rv)], axis=1).dropna().values

    return scaled_12d_raw, scaled_12d, scalers


def inverse_garch_12d_to_6d(generated_12d, scalers):
    """
    Diffusion이 생성한 12차원 데이터를 원본 6차원 수익률로 완벽하게 복원합니다.

    Args:
        generated_12d: (B, 12, T) 또는 (B, T, 12) 형태의 텐서/배열
        scalers_Z: Train에서 학습한 잔차(Z) 스케일러 리스트
        scalers_V: Train에서 학습한 로그변동성(V) 스케일러 리스트
    Returns:
        X_reconstructed: 복원된 (B, 6, T) 또는 (B, T, 6) 형태의 원본 6차원 텐서
    """
    # 3. 12차원 반갈죽 (앞 6: 잔차 Z, 뒤 6: 로그변동성 V)
    inv_12d = inverse_scaling(generated_12d.transpose(1, 2), scalers)

    Z_inv = inv_12d[:, :6, :]
    V_inv = inv_12d[:, 6:, :]

    # 5. 변동성 복구 (로그 해제)
    # 💡 방어 로직: 학습 초반 모델이 미친 값을 뱉을 때 exp가 우주로 가는 것 방지 (max 값을 약 10 내외로 제한)
    V_inv = np.clip(V_inv, a_min=-20.0, a_max=10.0)
    sigma_t = np.exp(V_inv)

    # 6. 최종 6차원 SDE 합성 (Z * sigma)
    # 평온했던 Z 노이즈에 변동성 스파이크를 때려 박아 현실 세계의 폭락/폭등을 구현
    X_reconstructed = Z_inv * sigma_t

    return X_reconstructed

In [3]:
# --- [0. 준비 단계] ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seq_len = 256

df = pd.read_csv(f"./data/indices2.csv")
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
df.set_index("Date", inplace=True)
df = df.apply(pd.to_numeric).astype(float)

rv = pd.read_csv(f"./data/RV_Parkinson.csv", index_col=0)
rv.index = pd.to_datetime(rv.index)
rv.index.name = "Date"

df_log_returns = np.log1p(df.pct_change()).dropna()
log_returns = df_log_returns.values
init_price_raw = np.array(df)[:-1, :]

# --- [1. Early Split (시간 순 분할)] ---
total_len = len(log_returns)
train_size = int(0.7 * total_len)

train_raw = df_log_returns.iloc[:train_size, :]
test_raw = df_log_returns.iloc[train_size:, :]  # Test는 원본 6차원 그대로 대기

train_init_raw = init_price_raw[:train_size]
test_init_raw = init_price_raw[train_size:]

# --- [2. GARCH 12차원 전처리 (Train ONLY)] ---
raw_12d, train_12d, scalers = fit_transform_garch_12d(df_log_returns, rv)
# scaled 행 0 = 원본 시계열 인덱스 ffd_burn (FFD burn-in 제거됨)

train_12d = train_12d[:train_size]

# --- [3. Windowing (시퀀스 쪼개기)] ---
# Train은 12차원 윈도우 생성
train_windowed = torch.from_numpy(rolling_window(train_12d, seq_len)).float()
train_init_windowed = (
    torch.from_numpy(train_init_raw[: -(seq_len - 1)]).float().unsqueeze(1)
)

test_windowed = torch.from_numpy(rolling_window(test_raw.values, seq_len)).float()
test_init_windowed = (
    torch.from_numpy(test_init_raw[: -(seq_len - 1)]).float().unsqueeze(1)
)

# (B, T, C) -> (B, C, T) 축 변환
training_data = train_windowed.clone().detach()  # (B, 256, 12)
test_data = test_windowed.clone().detach()  # (B, 256, 6)

training_set = TensorDataset(training_data)
test_set = TensorDataset(test_data)

train_dl = DataLoader(training_set, batch_size=256, shuffle=False)
test_dl = DataLoader(test_set, batch_size=256, shuffle=False)

print("\n🚀 데이터 준비 완료!")
print(f"Train DataLoader 배치 크기: {next(iter(train_dl))[0].shape}")
print(f"Test DataLoader 배치 크기: {next(iter(test_dl))[0].shape}")

c:\Users\Juhwankim\miniconda3\envs\research\lib\site-packages\pandas\core\internals\blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


Synced Shape - Returns: (5739, 6), RV: (5739, 6)
Final Cleaned Shape: (5720, 12)
Average Orthogonality Score (Corr): 0.0751

🚀 데이터 준비 완료!
Train DataLoader 배치 크기: torch.Size([256, 256, 12])
Test DataLoader 배치 크기: torch.Size([256, 256, 6])


# Ablation Studies

baseline(= RV decomposition(12d) + LambertW 포함 scaling + ModernTCN) 대비 한 요소만 제거/교체한 세 버전을 각각 학습·검증한 뒤, 마지막에 개별 결과를 뽑고 종합 테이블로 정리한다.

| 범주 | 변경점 | enc_in | backbone |
|------|--------|--------|----------|
| 1. w/o RV Decomposition | RV 분해 제외 → 원본 6차원 생성 | 6 | ModernTCN |
| 2. w/o LambertW | scaling 에서 LambertW(Gaussianize)만 제외 | 12 | ModernTCN |
| 3. w/o ModernTCN | backbone 을 quantTCN 으로 교체 | 12 | quantTCN |
| 4. 종합 | 개별 결과(원래 코드) + ablation 종합 테이블 CSV | - | - |

> **선행 조건**: 상단 Preprocessing 셀(0~3)을 먼저 실행해 `df_log_returns`, `rv`, `train_size`, `train_raw`, `test_data`, `train_init_raw`, `test_init_raw`, `train_12d`, `scalers` 가 정의돼 있어야 한다. 아래 공통 유틸 셀을 한 번 실행한 뒤 1→2→3→4 순서로 진행한다.

In [4]:
# ===== Ablation 공통 유틸 =====
# 학습/검증/생성/평가 코드는 baseline 셀(8,10,12,14,17,19,20)과 동일하며
# 전처리 / scaling / backbone 만 ablation 별로 주입한다.
# (Preprocessing 셀 0~3 실행 후 사용: df_log_returns, rv, train_size, train_raw,
#  test_data, train_init_raw, test_init_raw, train_12d, scalers 필요)
import os
import glob
import pandas as pd
from sklearn.preprocessing import StandardScaler
from models.modernTCN_cross_attention import ModernTCN
from models.quantTCN import QuantTCN

NUM_EPOCHS = 45
N_RUNS = 5
VAL_START = 25
TARGET_N = 2000  # 각 run 에서 (필터 생존) 확보할 합성 시퀀스 수
CLIP_LATENT = 8.0  # 역변환 전 latent clip 범위(±σ): LambertW exp overflow/ValueError 방지

ABL = {}  # ablation 설정 레지스트리 (각 섹션에서 채움) -> 섹션 4에서 일괄 생성/평가


# ---- (w/o LambertW용) Gaussianize(Lambert W) 단계만 제외한 scaling ----
def scaling_no_lambertw(data):
    scalers, scaled_data = [], []
    for i in range(data.shape[1]):
        feature_data = data[:, i].reshape(-1, 1)
        ss1, ss2 = StandardScaler(), StandardScaler()
        feature_scaled = ss2.fit_transform(ss1.fit_transform(feature_data))
        scalers.append((ss1, ss2, None))
        scaled_data.append(feature_scaled.flatten())
    return np.array(scaled_data).T, scalers


def inverse_scaling_no_lambertw(y, scalers):
    y = y.cpu().detach().numpy() if hasattr(y, "cpu") else y
    y_original = np.zeros_like(y)
    for idx in range(y.shape[1]):
        ss1, ss2, _ = scalers[idx]
        y_feature = y[:, idx, :]
        EPS = 1e-8
        y_feature = (y_feature - y_feature.mean(axis=0, keepdims=True)) / (
            y_feature.std(axis=0, keepdims=True) + EPS
        )
        orig_shape = y_feature.shape
        y_flat = y_feature.reshape(-1, 1)
        y_flat = ss1.inverse_transform(ss2.inverse_transform(y_flat))
        y_original[:, idx, :] = y_flat.reshape(orig_shape)
    return y_original


# ---- 12d -> 6d 복원 (scaling 역함수 주입형; 기존 inverse_garch_12d_to_6d 와 동일) ----
def garch_12d_to_6d(generated_12d, scalers, inv_scaling_fn=inverse_scaling):
    inv_12d = inv_scaling_fn(generated_12d.transpose(1, 2), scalers)
    Z_inv = inv_12d[:, :6, :]
    V_inv = np.clip(inv_12d[:, 6:, :], a_min=-20.0, a_max=10.0)
    return Z_inv * np.exp(V_inv)


# ---- (w/o RV Decomposition용) 원본 6d 로그수익률을 그대로 scaling ----
def fit_transform_no_rv(df_log_returns, scaling_fn=scaling):
    raw_6d = df_log_returns.dropna().values
    scaled_6d, scalers = scaling_fn(raw_6d)
    return raw_6d, scaled_6d, scalers


# ---- 12d 전처리 (scaling 함수 주입형; 기존 fit_transform_garch_12d 와 동일) ----
def fit_transform_garch_12d_with(
    df_log_returns, df_rv, scaling_fn=scaling, rv_window=20, min_periods=5
):
    eps = 1e-8
    common_idx = df_log_returns.index.intersection(df_rv.index)
    dlr, drv = df_log_returns.loc[common_idx], df_rv.loc[common_idx]
    future_rv = np.sqrt(
        drv.rolling(rv_window, min_periods=min_periods).mean().shift(-(rv_window - 1))
    )
    Z_t = dlr / (future_rv + eps)
    V_t = np.log(future_rv + eps)
    combined = pd.concat([Z_t, V_t], axis=1).dropna()
    scaled_12d, scalers = scaling_fn(combined.values)
    raw_12d = pd.concat([Z_t, np.log(future_rv)], axis=1).dropna().values
    return raw_12d, scaled_12d, scalers


# ---- backbone / diffusion 빌더 ----
def build_backbone(name, enc_in, max_seq_len):
    if name == "quanttcn":
        return QuantTCN(
            {
                "enc_in": enc_in,
                "hidden_dim": 128,
                "dilations": [1, 2, 4, 8, 16, 32],
                "kernel_size": 2,
                "dropout": 0.05,
                "use_time_emb": True,
                "time_emb_dim": 256,
            }
        )
    return ModernTCN(
        {
            "enc_in": enc_in,
            "dims": [128, 128, 128],
            "large_size": [15, 7, 5],
            "small_size": [3, 3, 3],
            "time_emb_dim": 256,
            "use_time_emb": True,
            "use_cross_attention": False,
            "max_seq_len": max_seq_len,
            "use_sequence_pos_encoding": False,
        }
    )


def build_diffusion(enc_in, backbone, seq_len_):
    ts_model = build_backbone(backbone, enc_in, seq_len_)
    betas = generate_cosine_schedule(75, 1.5, 1e-5, 0.5)
    diffusion = TimeSeriesGaussianDiffusion(
        ts_model,
        seq_len=seq_len_,
        num_channels=enc_in,
        betas=betas,
        loss_type="11",
        ema_decay=0.9999,
        ema_start=2000,
        ema_update_rate=1,
        sampling_type="student_t",
        prediction_type="epsilon",
    ).to(device)
    return ts_model, diffusion


# ---- 학습 (cell 8 + 10) ----
def train_ablation(tag, train_scaled, backbone, num_epochs=NUM_EPOCHS, batch_size=256):
    print(f"\n{'#'*70}\n# TRAIN [{tag}] backbone={backbone}, enc_in={train_scaled.shape[1]}\n{'#'*70}")
    training_data = torch.from_numpy(rolling_window(train_scaled, seq_len)).float()
    enc_in = training_data.shape[2]
    train_dl = DataLoader(
        TensorDataset(training_data.clone().detach()), batch_size=batch_size, shuffle=False
    )
    ts_model, diffusion = build_diffusion(enc_in, backbone, training_data.shape[1])
    optimizer = torch.optim.Adam(diffusion.parameters(), lr=2e-4, weight_decay=1e-5)
    weight_dir = file_manager.get_weight_path()
    os.makedirs(weight_dir, exist_ok=True)
    count_parameters(ts_model)

    clean_memory(device)
    pbar = tqdm(range(1, num_epochs + 1))
    for epoch in pbar:
        diffusion.train()
        epoch_loss, n_batches = 0.0, 0
        for batch in train_dl:
            x = batch[0].to(device)
            loss = diffusion(x)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            diffusion.update_ema()
            epoch_loss += loss.item()
            n_batches += 1
            del x, loss
        pbar.set_description(f"[{tag}] Epoch {epoch}")
        pbar.set_postfix(train_loss=f"{epoch_loss / n_batches:.4f}")
        torch.save(diffusion.state_dict(), f"{weight_dir}/{tag}_{epoch}.pt")
        if epoch % 10 == 0:
            clean_memory(device)
    return diffusion


# ---- 검증/Best 선정 (cell 12) ----
def validate_ablation(tag, diffusion, to_6d_fn, num_epochs=NUM_EPOCHS, val_start=VAL_START):
    weight_dir = file_manager.get_weight_path()
    result_path = file_manager.get_result_path(tag)
    os.makedirs(result_path, exist_ok=True)
    min_dist, best_epoch = float("inf"), -1
    real_data_val = (
        torch.from_numpy(rolling_window(train_raw, seq_len)).float().transpose(1, 2)
    )
    for epoch in tqdm(range(val_start, num_epochs + 1), desc=f"[{tag}] validate"):
        ckpt_path = f"{weight_dir}/{tag}_{epoch}.pt"
        if not os.path.exists(ckpt_path):
            continue
        diffusion.load_state_dict(torch.load(ckpt_path, map_location=device))
        diffusion.eval()
        with torch.no_grad():
            fake = diffusion.sample(batch_size=100, device=device, use_ema=True)
            fake = torch.clamp(fake, -CLIP_LATENT, CLIP_LATENT)
            with np.errstate(over="ignore", invalid="ignore"):
                fake_data = to_6d_fn(fake)
            emd = compute_avg_emd(real_data_val, fake_data, 100)
            if emd < min_dist:
                min_dist, best_epoch = emd, epoch
        del fake, fake_data
        clean_memory(device)

    if best_epoch == -1:
        print(f"❌ [{tag}] 유효한 체크포인트 없음")
        return -1
    print(f"🏆 [{tag}] Best Epoch: {best_epoch} (EMD: {min_dist:.5f})")
    best_ckpt_path = f"{weight_dir}/{tag}_{best_epoch}.pt"
    final_best_path = f"{result_path}/quantDiffusion_best({best_epoch}).pt"
    diffusion.load_state_dict(torch.load(best_ckpt_path, map_location=device))
    torch.save(diffusion.state_dict(), final_best_path)
    for fp in glob.glob(f"{weight_dir}/{tag}_*.pt"):
        if fp != best_ckpt_path:
            try:
                os.remove(fp)
            except Exception:
                pass
    print(f"💾 [{tag}] Best 모델 저장: {final_best_path}")
    return best_epoch


# ---- 생성 (cell 14 + 17): 목표 개수가 '생존'할 때까지 생성 ----
def generate_ablation(
    tag, enc_in, backbone, to_6d_fn, n_runs=N_RUNS, target_n=TARGET_N,
    gen_batch=100, max_total=2500,
):
    result_path = file_manager.get_result_path(tag)
    best_files = glob.glob(f"{result_path}/quantDiffusion_best*.pt")
    if not best_files:
        raise FileNotFoundError(f"[{tag}] best 가중치 없음 ({result_path}) - 먼저 학습/검증 실행")
    _, diffusion = build_diffusion(enc_in, backbone, seq_len)
    target_file = max(best_files, key=os.path.getmtime)
    diffusion.load_state_dict(torch.load(target_file, map_location=device))
    diffusion.to(device)
    diffusion.eval()
    print(f"✅ [{tag}] 로드: {os.path.basename(target_file)}")

    real_data = test_data.transpose(1, 2).cpu().detach().numpy()
    train_data_6d = (
        torch.from_numpy(rolling_window(train_raw, seq_len)).float().transpose(1, 2).numpy()
    )
    min_vals = train_data_6d.min(axis=(0, 2), keepdims=True)
    max_vals = train_data_6d.max(axis=(0, 2), keepdims=True)
    range_margin = (max_vals - min_vals) * 0.1
    lower_bound, upper_bound = min_vals - range_margin, max_vals + range_margin

    def _survivors(raw_list):
        # 역변환(inverse_scaling)은 배치 통계로 정규화하므로 전체 raw 를 한 번에 변환 후 필터.
        # LambertW 역변환의 exp overflow -> inf/nan 시퀀스는 isfinite 로 제거되고,
        # 아래 while 루프가 목표치를 채울 때까지 추가 생성(= 자동 재시도)한다.
        # 극단 outlier 가 inverse_scaling 내부 표준화를 오염시켜 LambertW exp 가
        # overflow(->inf, sklearn ValueError) 나는 것을 막기 위해 역변환 전 latent 를 clip.
        samples = torch.clamp(torch.cat(raw_list, dim=0), -CLIP_LATENT, CLIP_LATENT)
        with np.errstate(over="ignore", invalid="ignore"):
            f6 = to_6d_fn(samples)
        finite = np.isfinite(f6).all(axis=(1, 2))
        m = finite & np.all((f6 >= lower_bound) & (f6 <= upper_bound), axis=(1, 2))
        return f6[m]

    for run_idx in range(n_runs):
        # 1차: 생존율 ~60% 가정해 목표보다 넉넉히 생성
        n_init = int(np.ceil(target_n / 0.6 / gen_batch))
        raw = []
        with torch.no_grad():
            for _ in range(n_init):
                raw.append(diffusion.sample(batch_size=gen_batch, device=device, use_ema=True))
        survivors = _survivors(raw)
        # 부족하면 생존율 기반으로 추가 생성 (목표 달성까지, 상한 max_total)
        while len(survivors) < target_n and len(raw) * gen_batch < max_total:
            rate = max(len(survivors), 1) / (len(raw) * gen_batch)
            need = target_n - len(survivors)
            n_add = int(np.ceil(need / max(rate, 0.2) / gen_batch)) + 1
            with torch.no_grad():
                for _ in range(n_add):
                    raw.append(diffusion.sample(batch_size=gen_batch, device=device, use_ema=True))
            survivors = _survivors(raw)
            print(f"  [{tag}] run {run_idx}: 생성 {len(raw) * gen_batch} → 생존 {len(survivors)} (목표 {target_n})")
        clean_memory(device)

        if len(survivors) >= target_n:
            idx = np.random.choice(len(survivors), target_n, replace=False)
            final_fake_data = survivors[idx]
        else:
            print(f"⚠️ [{tag}] run {run_idx}: 상한 {max_total}개 생성에도 생존 {len(survivors)} < 목표 {target_n}. 있는 만큼 저장.")
            final_fake_data = survivors
        np.save(os.path.join(result_path, f"{run_idx}.npy"), final_fake_data)
        print(f"✅ [{tag}] Run {run_idx + 1}/{n_runs}: {final_fake_data.shape}")
    clean_memory(device)
    return real_data


# ---- 개별 평가: gen quality(cell 19) + portfolio(cell 20), 파일명만 tag ----
def evaluate_ablation(tag, real_data, n_runs=N_RUNS):
    result_path = file_manager.get_result_path(tag)
    windows = [1, 5, 20, 100]

    total_result_df = pd.DataFrame()
    for seed in range(n_runs):
        fake_data = np.load(f"{result_path}/{seed}.npy")
        result_df = pd.DataFrame()
        result_df = pd.concat([result_df, calculate_ccf(real_data, fake_data, 0)], axis=0)
        total_CCF_df = pd.DataFrame(columns=["Average"])
        for i in range(1, 11):
            total_CCF_df = pd.concat(
                [total_CCF_df, calculate_ccf(real_data, fake_data, i)], axis=1
            )
        result_df = pd.concat(
            [result_df, pd.DataFrame(total_CCF_df.mean(axis=1), columns=["Average"])], axis=0
        )
        result_df = pd.concat(
            [result_df, calculate_distribution_scores(real_data, fake_data, windows)], axis=0
        )
        result_df = pd.concat(
            [result_df, calculate_skew_kurtosis(real_data, fake_data, windows)], axis=0
        )
        _, acf_df = calculate_acf_score(real_data, fake_data, 40)
        result_df = pd.concat([result_df, acf_df], axis=0)
        _, leverage_df = calculate_leverage_effect(real_data, fake_data, 40)
        result_df = pd.concat([result_df, leverage_df], axis=0)
        total_result_df = pd.concat([total_result_df, result_df], axis=1)

    statistics_df = pd.DataFrame(index=total_result_df.index, columns=["MEAN", "STD"])
    statistics_df["MEAN"] = total_result_df.mean(axis=1).values
    statistics_df["STD"] = total_result_df.std(axis=1).values
    statistics_df.to_csv(file_manager.get_result_file(f"{tag}_gen_quality"))

    config_dir = f"{LOCAL_BASE_DIRECTORY}/utils/eval_portfolio/config.yaml"
    with open(config_dir) as file:
        eval_config = ml_collections.ConfigDict(yaml.safe_load(file))
    alphas = eval_config.alphas
    strat_names = ["equal_weight", "mean_reversion", "trend_following", "vol_trading"]
    strat_map = {
        "equal_weight": "EW",
        "mean_reversion": "MR",
        "trend_following": "TF",
        "vol_trading": "VT",
    }
    eval_size = real_data.shape[0]
    for seed in range(n_runs):
        fake_data = np.load(f"{result_path}/{seed}.npy")
        fake_data_torch = torch.from_numpy(fake_data).float().transpose(1, 2)
        real_data_torch = torch.from_numpy(real_data).float().transpose(1, 2)
        train_init_price = train_init_raw.reshape(-1, 1, 6).copy()
        train_init_price[:] = 100
        test_init_price = test_init_raw.reshape(-1, 1, 6).copy()
        test_init_price[:] = 100
        actual_eval_data = fake_data_torch[:eval_size]
        n_samples = actual_eval_data.shape[0]
        fake_prices = log_return_to_price(
            actual_eval_data,
            torch.tensor(train_init_price[:n_samples], dtype=torch.float32, device=actual_eval_data.device),
        )
        real_prices = log_return_to_price(
            real_data_torch[:n_samples],
            torch.tensor(test_init_price[:n_samples], dtype=torch.float32, device=actual_eval_data.device),
        )
        if not (fake_prices > 0).all():
            raise ValueError("Sanity Check Failed: Some fake prices are not positive.")
        final_results = {}
        with torch.no_grad():
            for alpha in alphas:
                eval_config.Evaluation.TestMetrics.var.alpha = alpha
                eval_config.Evaluation.TestMetrics.es.alpha = alpha
                var_row, es_row = {}, {}
                for strat_name in strat_names:
                    subres_dict, _, _ = full_evaluation(
                        fake_prices, real_prices, eval_config, strat_name=strat_name
                    )
                    sn = strat_map[strat_name]
                    var_row[sn] = round(subres_dict.get("var_rel_mean", 0.0), 4)
                    es_row[sn] = round(subres_dict.get("es_rel_mean", 0.0), 4)
                var_row["Average"] = round(np.mean(list(var_row.values())), 4)
                es_row["Average"] = round(np.mean(list(es_row.values())), 4)
                final_results[f"RE_VaR({alpha})"] = var_row
                final_results[f"RE_ES({alpha})"] = es_row
        ordered_keys = [f"{m}({a})" for m in ["RE_VaR", "RE_ES"] for a in alphas]
        df_final = pd.DataFrame.from_dict(final_results, orient="index")
        cols_order = [strat_map[s] for s in strat_names] + ["Average"]
        df_final = df_final.reindex(index=ordered_keys, columns=cols_order)
        df_final.index.name = "Metric"
        df_final.T.to_csv(file_manager.get_result_file(f"{tag}_{seed}_portfolio"))
    print(f"✅ [{tag}] 개별 결과 저장: {tag}_gen_quality.csv, {tag}_<seed>_portfolio.csv")


# ---- 종합 테이블 (baseline + ablation 들을 한 표로) ----
def _pm(m, s, nd=4):
    return f"{m:.{nd}f} ± {s:.{nd}f}" if (np.isfinite(m) and np.isfinite(s)) else ""


def aggregate_gen_quality(tag_label_map, out_name="ablation_summary_gen_quality"):
    cols = {}
    for tag, label in tag_label_map.items():
        f = file_manager.get_result_file(f"{tag}_gen_quality")
        if not os.path.exists(f):
            print(f"  (skip {tag}: {os.path.basename(f)} 없음)")
            continue
        df = pd.read_csv(f, index_col=0)
        cols[label] = pd.Series(
            [_pm(m, s) for m, s in zip(df["MEAN"], df["STD"])], index=df.index
        )
    summary = pd.DataFrame(cols)
    summary.to_csv(file_manager.get_result_file(out_name), encoding="utf-8-sig")
    print("saved:", file_manager.get_result_file(out_name))
    return summary


def aggregate_portfolio(tag_label_map, out_name="ablation_summary_portfolio", n_runs=N_RUNS):
    cols = {}
    for tag, label in tag_label_map.items():
        parts = []
        for seed in range(n_runs):
            f = file_manager.get_result_file(f"{tag}_{seed}_portfolio")
            if os.path.exists(f):
                parts.append(pd.read_csv(f, index_col=0))
        if not parts:
            print(f"  (skip {tag}: portfolio csv 없음)")
            continue
        stacked = pd.concat(parts, axis=0)
        avg_rows = stacked.loc[["Average"]]  # 'Average' 전략 행 (seed 개수만큼)
        m, sd = avg_rows.mean(axis=0), avg_rows.std(axis=0)
        cols[label] = pd.Series([_pm(a, b) for a, b in zip(m, sd)], index=m.index)
    summary = pd.DataFrame(cols)
    summary.to_csv(file_manager.get_result_file(out_name), encoding="utf-8-sig")
    print("saved:", file_manager.get_result_file(out_name))
    return summary


# ---- 커널 재시작 후 '학습 없이' ABL 재구성 (저장된 weight 로 생성/평가용) ----
def build_abl_registry():
    """전처리/scaling 만 다시 수행해 ABL 설정(enc_in, backbone, to_6d_fn)을 복원한다.
    scaler 는 데이터에 대해 결정적이므로 학습 때와 동일하게 재현된다 (재학습 불필요).
    weight 는 result_path 에 저장된 quantDiffusion_best*.pt 를 generate_ablation 이 로드한다.
    (Preprocessing 셀 0~3 필요: df_log_returns, rv, train_12d, scalers)"""
    reg = {}
    _, _, sc_wo_rv = fit_transform_no_rv(df_log_returns, scaling_fn=scaling)
    reg["wo_rv"] = dict(
        label="w/o RV Decomp.", enc_in=6, backbone="moderntcn",
        to_6d_fn=lambda s, sc=sc_wo_rv: inverse_scaling(s.transpose(1, 2), sc),
    )
    _, _, sc_wo_lw = fit_transform_garch_12d_with(
        df_log_returns, rv, scaling_fn=scaling_no_lambertw
    )
    reg["wo_lambertw"] = dict(
        label="w/o LambertW", enc_in=12, backbone="moderntcn",
        to_6d_fn=lambda s, sc=sc_wo_lw: garch_12d_to_6d(s, sc, inv_scaling_fn=inverse_scaling_no_lambertw),
    )
    reg["wo_moderntcn"] = dict(
        label="w/o ModernTCN", enc_in=12, backbone="quanttcn",
        to_6d_fn=lambda s, sc=scalers: garch_12d_to_6d(s, sc, inv_scaling_fn=inverse_scaling),
    )
    return reg

## 1. w/o RV Decomposition
RV 분해(Z·log-vol 12차원) 없이 원본 6차원 로그수익률을 그대로 학습한다. scaling 은 baseline 과 동일(LambertW 포함). 6차원을 생성해 6차원으로 곧바로 역변환.

In [5]:
# RV decomposition 없이 원본 6차원 로그수익률을 그대로 학습/생성 (scaling 은 baseline 과 동일, LambertW 포함)
_, scaled6_full, scalers_wo_rv = fit_transform_no_rv(df_log_returns, scaling_fn=scaling)
train6 = scaled6_full[:train_size]

ABL["wo_rv"] = dict(
    label="w/o RV Decomp.",
    train_scaled=train6,
    enc_in=6,
    backbone="moderntcn",
    to_6d_fn=lambda s: inverse_scaling(s.transpose(1, 2), scalers_wo_rv),
)

# 학습 + 검증(Best epoch 선정)
diff_wo_rv = train_ablation("wo_rv", train6, "moderntcn")
validate_ablation("wo_rv", diff_wo_rv, ABL["wo_rv"]["to_6d_fn"])


######################################################################
# TRAIN [wo_rv] backbone=moderntcn, enc_in=6
######################################################################


c:\Users\Juhwankim\Documents\Github\quantDiffusion\modules\diffusion.py:126: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  to_torch = lambda x: torch.tensor(x, dtype=torch.float32)


KeyboardInterrupt: 

## 2. w/o LambertW
RV 분해(12차원)는 baseline 과 동일하게 유지하고, scaling 에서 LambertW(Gaussianize) 단계만 제거(`SS1 → SS2`)한다.

In [ ]:
# RV decomposition(12차원)은 baseline 과 동일, scaling 에서 LambertW(Gaussianize) 단계만 제외 (SS1 -> SS2)
_, train12_wo_lw_full, scalers_wo_lw = fit_transform_garch_12d_with(
    df_log_returns, rv, scaling_fn=scaling_no_lambertw
)
train12_wo_lw = train12_wo_lw_full[:train_size]

ABL["wo_lambertw"] = dict(
    label="w/o LambertW",
    train_scaled=train12_wo_lw,
    enc_in=12,
    backbone="moderntcn",
    to_6d_fn=lambda s: garch_12d_to_6d(s, scalers_wo_lw, inv_scaling_fn=inverse_scaling_no_lambertw),
)

# 학습 + 검증(Best epoch 선정)
diff_wo_lw = train_ablation("wo_lambertw", train12_wo_lw, "moderntcn")
validate_ablation("wo_lambertw", diff_wo_lw, ABL["wo_lambertw"]["to_6d_fn"])

## 3. w/o ModernTCN
전처리·scaling 은 baseline 과 동일하게 두고 backbone 만 ModernTCN → quantTCN 으로 교체한다.

In [ ]:
# backbone 을 ModernTCN -> quantTCN 으로 교체 (전처리/scaling 은 baseline 그대로: train_12d, scalers 재사용)
ABL["wo_moderntcn"] = dict(
    label="w/o ModernTCN",
    train_scaled=train_12d,
    enc_in=12,
    backbone="quanttcn",
    to_6d_fn=lambda s: garch_12d_to_6d(s, scalers, inv_scaling_fn=inverse_scaling),
)

# 학습 + 검증(Best epoch 선정)
diff_wo_mtcn = train_ablation("wo_moderntcn", train_12d, "quanttcn")
validate_ablation("wo_moderntcn", diff_wo_mtcn, ABL["wo_moderntcn"]["to_6d_fn"])

## 4. 개별 결과 + Ablation 종합 테이블
1~3 에서 선정된 Best 가중치로 (원래 코드 그대로) 각 ablation 의 합성데이터·개별 평가 CSV 를 만들고, 마지막에 baseline 포함 종합 테이블 CSV(`ablation_summary_gen_quality.csv`, `ablation_summary_portfolio.csv`)로 한 번에 정리한다.

In [5]:
# ===== 4-1. 개별 결과: Best 가중치로 각 ablation 합성데이터 생성 + 개별 평가 =====
# ABL 이 비어 있으면(커널 재시작 등) 저장된 weight 기반으로 '학습 없이' 재구성한다.
# (Preprocessing 셀 0~3 은 먼저 실행돼 있어야 함: df_log_returns, rv, train_12d, scalers)
if not ABL:
    print("ABL 이 비어 있어 build_abl_registry() 로 재구성합니다 (학습 생략, 저장된 weight 사용).")
    ABL = build_abl_registry()
    for _t, _c in ABL.items():
        _bf = glob.glob(f"{file_manager.get_result_path(_t)}/quantDiffusion_best*.pt")
        print(f"  - {_t:14s} backbone={_c['backbone']:9s} "
              f"weight={'O ' + os.path.basename(_bf[0]) if _bf else 'X (학습 필요)'}")

for tag, cfg in ABL.items():
    print(f"\n{'='*70}\n개별 결과 생성/평가: {cfg['label']} ({tag})\n{'='*70}")
    real_data = generate_ablation(tag, cfg["enc_in"], cfg["backbone"], cfg["to_6d_fn"])
    evaluate_ablation(tag, real_data)

ABL 이 비어 있어 build_abl_registry() 로 재구성합니다 (학습 생략, 저장된 weight 사용).
  - wo_rv          backbone=moderntcn weight=O quantDiffusion_best(26).pt
  - wo_lambertw    backbone=moderntcn weight=O quantDiffusion_best(26).pt
  - wo_moderntcn   backbone=quanttcn  weight=O quantDiffusion_best(28).pt

개별 결과 생성/평가: w/o RV Decomp. (wo_rv)


c:\Users\Juhwankim\Documents\Github\quantDiffusion\modules\diffusion.py:126: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  to_torch = lambda x: torch.tensor(x, dtype=torch.float32)


✅ [wo_rv] 로드: quantDiffusion_best(26).pt
✅ [wo_rv] Run 1/5: (2000, 6, 256)
✅ [wo_rv] Run 2/5: (2000, 6, 256)
✅ [wo_rv] Run 3/5: (2000, 6, 256)
✅ [wo_rv] Run 4/5: (2000, 6, 256)
✅ [wo_rv] Run 5/5: (2000, 6, 256)
49
49
61
63
77
98
49
49
61
63
77
98
49
49
61
63
77
98
49
60
61
66
77
103
49
60
61
66
77
103
49
60
61
66
77
103
49
72
61
72
77
96
49
72
61
72
77
96
49
72
61
72
77
96
49
49
61
69
77
91
49
49
61
69
77
91
49
49
61
69
77
91
49
49
61
63
77
104
49
49
61
63
77
104
49
49
61
63
77
104
✅ [wo_rv] 개별 결과 저장: wo_rv_gen_quality.csv, wo_rv_<seed>_portfolio.csv

개별 결과 생성/평가: w/o LambertW (wo_lambertw)
✅ [wo_lambertw] 로드: quantDiffusion_best(26).pt


c:\Users\Juhwankim\Documents\Github\quantDiffusion\modules\diffusion.py:126: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  to_torch = lambda x: torch.tensor(x, dtype=torch.float32)


✅ [wo_lambertw] Run 1/5: (2000, 6, 256)
✅ [wo_lambertw] Run 2/5: (2000, 6, 256)
✅ [wo_lambertw] Run 3/5: (2000, 6, 256)
✅ [wo_lambertw] Run 4/5: (2000, 6, 256)
✅ [wo_lambertw] Run 5/5: (2000, 6, 256)
49
55
61
57
77
70
49
55
61
57
77
70
49
55
61
57
77
70
49
50
61
60
77
73
49
50
61
60
77
73
49
50
61
60
77
73
49
57
61
60
77
61
49
57
61
60
77
61
49
57
61
60
77
61
49
55
61
49
77
77
49
55
61
49
77
77
49
55
61
49
77
77
49
85
61
55
77
68
49
85
61
55
77
68
49
85
61
55
77
68
✅ [wo_lambertw] 개별 결과 저장: wo_lambertw_gen_quality.csv, wo_lambertw_<seed>_portfolio.csv

개별 결과 생성/평가: w/o ModernTCN (wo_moderntcn)
✅ [wo_moderntcn] 로드: quantDiffusion_best(40).pt


c:\Users\Juhwankim\Documents\Github\quantDiffusion\modules\diffusion.py:126: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  to_torch = lambda x: torch.tensor(x, dtype=torch.float32)


✅ [wo_moderntcn] Run 1/5: (2000, 6, 256)
✅ [wo_moderntcn] Run 2/5: (2000, 6, 256)
✅ [wo_moderntcn] Run 3/5: (2000, 6, 256)
✅ [wo_moderntcn] Run 4/5: (2000, 6, 256)
✅ [wo_moderntcn] Run 5/5: (2000, 6, 256)
49
73
61
69
77
96
49
73
61
69
77
96
49
73
61
69
77
96
49
55
61
50
77
90
49
55
61
50
77
90
49
55
61
50
77
90
49
54
61
53
77
91
49
54
61
53
77
91
49
54
61
53
77
91
49
63
61
67
77
93
49
63
61
67
77
93
49
63
61
67
77
93
49
77
61
56
77
91
49
77
61
56
77
91
49
77
61
56
77
91
✅ [wo_moderntcn] 개별 결과 저장: wo_moderntcn_gen_quality.csv, wo_moderntcn_<seed>_portfolio.csv


In [6]:
# ===== 4-2. Ablation 종합 테이블 — 저장된 개별 결과 CSV 만으로 동작 =====
# ABL(메모리)에 의존하지 않는 고정 매핑이라, 커널 재시작 후 '이 셀만' 단독 실행해도
# 디스크의 {tag}_gen_quality.csv / {tag}_<seed>_portfolio.csv 를 모아 종합표를 만든다.
# (없는 tag 는 자동 skip. baseline(student_t) 은 1_modeling_RV 결과가 있으면 포함됨)
tag_label_map = {
    "student_t": "Baseline",
    "wo_rv": "w/o RV Decomp.",
    "wo_lambertw": "w/o LambertW",
    "wo_moderntcn": "w/o ModernTCN",
}

# 어떤 tag 의 결과가 실제로 디스크에 있는지 점검 (X 면 그 ablation 의 4-1 미실행 -> skip)
print("결과 존재 여부 (X 면 해당 ablation 은 4-1 생성/평가 필요):")
for tag, label in tag_label_map.items():
    gq = os.path.exists(file_manager.get_result_file(f"{tag}_gen_quality"))
    pf = os.path.exists(file_manager.get_result_file(f"{tag}_0_portfolio"))
    print(f"  - {label:20s}: gen_quality={'O' if gq else 'X'}  portfolio={'O' if pf else 'X'}")

print("\n[Ablation 종합 - Generation Quality]")
summary_gen_quality = aggregate_gen_quality(tag_label_map)
print(summary_gen_quality)

print("\n[Ablation 종합 - Portfolio (Average 전략)]")
summary_portfolio = aggregate_portfolio(tag_label_map)
print(summary_portfolio)

결과 존재 여부 (X 면 해당 ablation 은 4-1 생성/평가 필요):
  - Baseline            : gen_quality=O  portfolio=O
  - w/o RV Decomp.      : gen_quality=O  portfolio=O
  - w/o LambertW        : gen_quality=O  portfolio=O
  - w/o ModernTCN       : gen_quality=O  portfolio=O

[Ablation 종합 - Generation Quality]
saved: c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/statistics/ablation_summary_gen_quality.csv
                        Baseline   w/o RV Decomp.     w/o LambertW  \
CCF(τ = 0)_MAE   0.0367 ± 0.0003  0.0491 ± 0.0003  0.0389 ± 0.0002   
CCF(τ = 0)_FROB  0.3482 ± 0.0014  0.3665 ± 0.0017  0.3140 ± 0.0014   
CCF(τ = 0)_MSE   0.0034 ± 0.0001  0.0037 ± 0.0001  0.0027 ± 0.0001   
CCF(τ > 0)_MAE   0.0314 ± 0.0001  0.0353 ± 0.0001  0.0329 ± 0.0002   
CCF(τ > 0)_FROB  0.6418 ± 0.0014  0.6530 ± 0.0013  0.6258 ± 0.0034   
CCF(τ > 0)_MSE   0.0019 ± 0.0000  0.0020 ± 0.0000  0.0018 ± 0.0000   
EMD(1)           0.0014 ± 0.0000  0.0013 ± 0.0000  0.0015 ± 0.0000   
EMD(5)           0.0036 ± 0.0000  0.0034